**STEP 1: Importing dataset**

TinyStories is a synthetic dataset of short stories that only contain words typically used for 3-4 years old. We can get the data from HuggingFace.

In [ ]:
!pip install datasets

In [ ]:
from datasets import  load_dataset
ds= load_dataset("roneneldan/TinyStories")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

**STEP 2: Tokenize the Dataset**

In this step, we are going to do following things:

1. Tokenize the dataset into tolken IDs

2. Create a file called "train.bin" and "validation.bin" where we will store the token IDs from the entire dataset

3. We make sure the IDs are stored on the disk, rather than RAM for efficient computation

In [ ]:
!pip install tiktoken
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

enc= tiktoken.get_encoding("gpt2")

def process(example):
  ids= enc.encode_ordinary(example['text']) #encode_ordinary ignores any special tokens
  out=  {'ids': ids, 'len': len(ids)}
  return out

if not os.path.exists("train.bin"):
  tokenized= ds.map(
      process,
      remove_columns=['text'],
      desc= "tokenizing the splits",
      num_proc=8,
  )
#concatenate all the ids in each dataset into a large file we can use for traing
for split, dset in tokenized.items():
  arr_len= np.sum(dset['len'], dtype= np.uint64)
  filename= f"{split}.bin"
  dtype= np.uint16  #Can do since  enc.max_token_value=50256 < 2**16
  arr= np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
  total_batches= 1024

idx=0
for batch_idx in tqdm(range (total_batches), desc= f"writing {filename}"):
  #batch together samples for faster write
  batch= dset.shard(num_shards= total_batches, index=batch_idx, contiguous= True).with_format('numpy')
  arr_batch= np.concatenate(batch['ids'])
  #write into mmap
  arr[idx : idx + len(arr_batch)]= arr_batch
  idx += len(arr_batch)
arr.flush()



tokenizing the splits (num_proc=8):   0%|          | 0/2119719 [00:00<?, ? examples/s]

tokenizing the splits (num_proc=8):   0%|          | 0/21990 [00:00<?, ? examples/s]

writing validation.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

**STEP 3: Create INput-Output batches for the dataset**

In [ ]:
import torch
def get_batch(split):

    if split == "train":
        filename = "train.bin"
    elif split == "validation":
        filename = "validation.bin"
    else:
        raise ValueError("Invalid split")

    # Load the memmap - ensures 'data' is always defined
    data = np.memmap(filename, dtype=np.uint16, mode='r')

    # Generate indices
    # block_size and batch_size must be defined in your global scope
    ix = torch.randint(0, len(data) - block_size, (batch_size,))

    # Extract and cast to int64
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    # Move to device
    if device_type == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)

    return x, y

**STEP 4: Define the SLM Model Architecture**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm.auto import tqdm
from contextlib import nullcontext
import os


class LayerNorm(nn.Module):
  def __init__(self, ndim, bias):
    super().__init__()
    self.weight= nn.Parameter(torch.ones(ndim))
    self.bias= nn.Parameter(torch.zeros(ndim)) if bias else None
  def forward(self, x):
    return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)


class CasualSelfAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    assert config.n_embd % config.n_head== 0
    self.c_attn= nn.Linear(config.n_embd, 3*config.n_embd, bias= config.bias)
    self.c_proj= nn.Linear(config.n_embd, config.n_embd, bias= config.bias)
    self.attn_dropout= nn.Dropout(config.dropout)
    self.resid_dropout= nn.Dropout(config.dropout)
    self.n_head= config.n_head
    self.n_embd= config.n_embd
    self.flash= hasattr(F, "scaled_dot_product_attention")
    if not self.flash:
      self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))

  def forward(self, x):
    B, T, C= x.size()
    q, k, v= self.c_attn(x).split(self.n_embd, dim=2)
    k= k.view(B, T, self.n_head, C// self.n_head). transpose(1, 2)
    q= q.view(B, T, self.n_head, C// self.n_head). transpose(1, 2)
    v= v.view(B, T, self.n_head, C// self.n_head). transpose(1, 2)

    if self.flash:
      y= F.scaled_dot_product_attention(q, k, v, attn_mask= None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal= True)
    else:
      att= (q @ k.tranpose(-2,-1))* (1.0 / math.sqrt(k.size(-1)))
      att= att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
      att= F.softmax(att, dim=-1)
      att= self.attn_dropout(att)
      y= att @ v

    y= y.transpose(1,2).contiguous().view(B, T, C)
    y= self.resid_dropout(self.c_proj(y))
    return y

class MLP(nn.Module): #multi layer perceptron block
  def __init__(self, config):
    super().__init__()
    self.c_fc= nn.Linear(config.n_embd, 4* config.n_embd, bias=config.bias)
    self.gelu= nn.GELU()
    self.c_proj= nn.Linear(4* config.n_embd, config.n_embd, bias= config.bias)
    self.dropout= nn.Dropout(config.dropout)
  def forward(self, x):
    return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))
#Transformer block, arraged everything together
class Block(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.ln1 = LayerNorm(config.n_embd, config.bias)
    self.attn = CasualSelfAttention(config)
    self.ln2 = LayerNorm(config.n_embd, config.bias)
    self.mlp= MLP(config)
  def forward(self, x):
    x = x + self.attn(self.ln1(x))
    x = x + self.mlp(self.ln2(x))
    return x

@dataclass
class GPTConfig:
  block_size: int
  vocab_size: int
  n_layer: int
  n_head: int
  n_embd: int
  dropout: float = 0.0
  bias: bool= True

class GPT(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.config= config
    self.transformer= nn.ModuleDict(dict(
        wte= nn.Embedding(config.vocab_size, config.n_embd),
        wpe= nn.Embedding(config.block_size, config.n_embd),
        drop= nn.Dropout(config.dropout),
        h= nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
        ln_f=LayerNorm(config.n_embd, config.bias),
    ))
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias= False)
    self.transformer.wte.weight= self.lm_head.weight # weight tying

    self.apply(self._init_weights)
    for pn, p in self.named_parameters():
      if pn.endswith('c_proj.weight'):
        nn.init.normal_(p, mean= 0.0, std= 0.02/ math.sqrt(2* config.n_layer))

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      nn.init.normal_(module.weight, mean=0.0, std= 0.02)
      if module.bias is not None:
        nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      nn.init.normal_(module.weight, mean= 0.0, std=0.02)

  def forward(self, idx, targets=None):
    device= idx.device
    b, t= idx.size()
    assert t<=self.config.block_size
    pos = torch.arange(0, t, dtype=torch.long, device=device)

    tok_emb= self.transformer.wte(idx)
    pos_emb= self.transformer.wpe(pos)
    x= self.transformer.drop(tok_emb + pos_emb)
    for block in self.transformer.h:
      x= block(x)
    x= self.transformer.ln_f(x)

    if targets is not None:
      logits= self.lm_head(x)
      loss= F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index= -1)
      return logits, loss
    else:
      logits= self.lm_head(x[:, [-1], :])
      return logits, None

  @torch.no_grad()
  def generate(self, idx, max_new_tokens, temperature= 1.0, top_k=None):
    """
    Generate tokens given a conditioning sequence.
    idx: Tensor of shape (B, T)
    """
    for _ in range(max_new_tokens):
      idx_cond= idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
      logits, _ = self(idx_cond)
      logits= logits[:, -1, :] / temperature
      if top_k is not None:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float('Inf')
      probs= F.softmax(logits, dim=-1)
      idx_next= torch.multinomial(probs, num_samples= 1)
      idx= torch.cat((idx, idx_next), dim=1)
    return idx




In [ ]:
config = GPTConfig(
    vocab_size= 50257,
    block_size=128,
    n_layer=6, #number of transformers
    n_head=6, #number of attention heads in each block
    n_embd=384, #embedding dimension
    dropout=0.1,
    bias=True
)

model = GPT(config)

**STEP 5: Define the loss function**

In [ ]:
def estimate_loss(model):
  out = {}
  model.eval()
  with torch.inference_mode():
    for split in ['train', 'validation']:
      losses = torch.zeros(eval_iters)
      for k in range(eval_iters):
        X, Y= get_batch(split)
        with ctx:
          logits, loss = model(X, Y)
        losses[k] = loss.item()
      out[split] = losses.mean()
  model.train()
  return out

**STEP 6: Define SLM Training Configuration** (Part 1)

In [ ]:
# Training Config
import torch
from contextlib import nullcontext

learning_rate = 1e-4 #more stable training, earlier 1e-4
max_iters = 20000 #increase from 25000
warmup_steps = 1000 #smoother initial train, earlier 100
min_lr = 1e-5 #lower rate, earlier 5e-4
eval_iters = 500 #increased from 100
batch_size = 32 #changed from 16, better gradient estimate
block_size = 128 #changed from 64, capture longer range dependencies

gradient_accumulation_steps = 32 #reduced from 50

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device_type = 'cuda' if 'cuda' in device else 'cpu' #for later use in torch.autocast
#note: float16 data type will automatically use GradScaler

dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

torch.set_default_device(device)
torch.manual_seed(42)

**STEP 7: Define SLM Training Configuration** (Part 2)

In [ ]:
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

##PUT IN WEIGHT DECAY, CHANGED BETA2 TO 0.95
optimizer = torch.optim.AdamW(model.parameters(), lr= learning_rate, betas= (0.9, 0.95), weight_decay=0.1, eps=1e-9)

scheduler_warmup = LinearLR(optimizer, total_iters = warmup_steps) #Tmplement linear warmup
scheduler_decay = CosineAnnealingLR(optimizer, T_max= max_iters - warmup_steps, eta_min = min_lr) #Implement lr decay
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps])

# Modern approach
scaler = torch.amp.GradScaler('cuda', enabled=(dtype == 'float16'))

/tmp/ipykernel_3683/3247705702.py:11: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler('cuda', enabled=(dtype == 'float16'))


**STEP 8: Pre-train the SLM**

In [ ]:
best_val_loss = float('inf')
best_model_params_path = 'best_model_params.pt'
train_loss_list, validation_loss_list =[], []

#Ensure model is on the correct device
model = model.to(device)

#In your traing loop
for epoch in tqdm(range(max_iters)):
  if epoch % eval_iters == 0 and epoch != 0:
    #ensure estimate_loss uses the correct device
    losses = estimate_loss(model)
    print(f"Epoch {epoch}: train loss {losses['train']: .4f}, val loss {losses['validation']: .4f}")
    print(f"The current learning rate: {optimizer.param_groups[0]['lr']: .5f}")
    train_loss_list += [losses['train']]
    validation_loss_list += [losses['validation']]

    if losses['validation'] < best_val_loss:
      best_val_loss = losses['validation']
      torch.save(model.state_dict(), best_model_params_path)

  #Ensure X and y are on the correct device
  X, y = get_batch("train")
  X, y = X.to(device), y.to(device)

  with ctx:
    logits, loss = model(X, y)
    loss = loss / gradient_accumulation_steps
    scaler.scale(loss).backward()

  if ((epoch + 1) % gradient_accumulation_steps == 0) or (epoch +1 == max_iters):
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

  scheduler.step()

  0%|          | 0/20000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:1195: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


**STEP 9: Plot the SLM Loss Function**

In [ ]:
import matplotlib.pyplot as plt
train_loss_list_converted = [i.cpu().detach() for i in train_loss_list]
validation_loss_list_converted = [i.cpu().detach() for i in validation_loss_list]

plt.plot(train_loss_list_converted, 'g', label='train_loss')
plt.plot(validation_loss_list_converted, 'r', label='validation_loss')
plt.xlabel("Steps - Every 100 epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()
# Inside estimate_loss
X, Y = get_batch(split)
print(f"DEBUG: {split} data shape: {X.shape}")


**STEP 10: Run SLM Inference on our trained model**

In [ ]:
#Load the model
model = GPT(config) #recreate the model with same config
device = 'cuda' if torch.cuda.is_available() else 'cpu'
best_model_params_path = 'best_model_params.pt'
model.load_state_dict(
    torch.load(best_model_params_path, map_location=device)
)

model = model.to(device)
model.eval()

In [ ]:
sentence = "Once upon a time there was a pumpkin."

context = torch.tensor(
    enc.encode_ordinary(sentence)
).unsqueeze(0).to(device)

with torch.no_grad():
    y = model.generate(context, 200)

print(enc.decode(y.squeeze().tolist()))

In [ ]:
import os

files = [
    "train.bin",
    "validation.bin",
    "best_model_params.pt"
]

for file in files:
    if os.path.exists(file):
        os.remove(file)
        print(f"Deleted: {file}")
    else:
        print(f"Not found: {file}")